**Project 1**

**Setting up data:**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import (MinMaxScaler,MaxAbsScaler, RobustScaler)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


features = [
    "matchId",
    "blueTeamControlWardsPlaced", "blueTeamWardsPlaced", "blueTeamTotalKills", "blueTeamDragonKills",
    "blueTeamHeraldKills", "blueTeamTowersDestroyed", "blueTeamInhibitorsDestroyed",
    "blueTeamTurretPlatesDestroyed", "blueTeamFirstBlood", "blueTeamMinionsKilled",
    "blueTeamJungleMinions", "blueTeamTotalGold", "blueTeamXp", "blueTeamTotalDamageToChamps",
    "redTeamControlWardsPlaced", "redTeamWardsPlaced", "redTeamTotalKills", "redTeamDragonKills",
    "redTeamHeraldKills", "redTeamTowersDestroyed", "redTeamInhibitorsDestroyed",
    "redTeamTurretPlatesDestroyed", "redTeamMinionsKilled", "redTeamJungleMinions",
    "redTeamTotalGold", "redTeamXp", "redTeamTotalDamageToChamps",
    "blueWin", "temp"
]

data = pd.read_csv('data.csv', names = features, na_values=["?", "N/A", ""], skipinitialspace=True, skiprows=1)

data_without_classification = data.drop(columns=["matchId","blueWin", "temp"])
results_column = data["blueWin"]



X_train, X_test, y_train, y_test = train_test_split(
    data_without_classification, results_column,
    test_size=0.20,
    stratify=results_column,
    random_state=42,
)

print("train:", len(X_train))
print("test:", len(X_test))

train: 19380
test: 4845


**Benchmark we will compare our model with**

In [2]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Model Test Accuracy: {accuracy * 100:.2f}%")
print(f"Train: {model.score(X_train, y_train) * 100:.2f}%")
print(f"Test:  {model.score(X_test, y_test) * 100:.2f}%")

Model Test Accuracy: 66.44%
Train: 100.00%
Test:  66.44%


**Data Representation**

We don't have imbalanced data because of this:

In [3]:
data['blueWin'].value_counts(normalize=True)

blueWin
0    0.50547
1    0.49453
Name: proportion, dtype: float64

**Missing Values**

No missing values:

In [4]:
data.isna().sum()

matchId                          0
blueTeamControlWardsPlaced       0
blueTeamWardsPlaced              0
blueTeamTotalKills               0
blueTeamDragonKills              0
blueTeamHeraldKills              0
blueTeamTowersDestroyed          0
blueTeamInhibitorsDestroyed      0
blueTeamTurretPlatesDestroyed    0
blueTeamFirstBlood               0
blueTeamMinionsKilled            0
blueTeamJungleMinions            0
blueTeamTotalGold                0
blueTeamXp                       0
blueTeamTotalDamageToChamps      0
redTeamControlWardsPlaced        0
redTeamWardsPlaced               0
redTeamTotalKills                0
redTeamDragonKills               0
redTeamHeraldKills               0
redTeamTowersDestroyed           0
redTeamInhibitorsDestroyed       0
redTeamTurretPlatesDestroyed     0
redTeamMinionsKilled             0
redTeamJungleMinions             0
redTeamTotalGold                 0
redTeamXp                        0
redTeamTotalDamageToChamps       0
blueWin             

**Model Engineering / K Nearest neighbours**

In [5]:
pipe = Pipeline([("scaling_methods", StandardScaler()), ("knn", KNeighborsClassifier())])

params = {
    "scaling_methods": [
        StandardScaler(),
        MinMaxScaler(),
        MaxAbsScaler(),
         RobustScaler(),
        "passthrough"
    ],
    "knn__n_neighbors": [3, 6, 8, 12, 18, 30, 40, 50,60,70,80, 90 , 150, 200],
    "knn__weights": ["uniform", "distance"]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

predictions = grid.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

Best parameters:
{'knn__n_neighbors': 200, 'knn__weights': 'distance', 'scaling_methods': StandardScaler()}
Best cross-validation accuracy:
0.7560887512899896
Test accuracy: 0.7606


**Model Engineering / Logistic Regression**

In [6]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid.fit(X_train, y_train)

predictions = grid.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")


Test accuracy: 0.7626


**Model Engineering / Decission Tree**

In [7]:
pipe = Pipeline([("dt", DecisionTreeClassifier())])

params = {
    "dt__criterion": ["gini", "entropy"],
    "dt__max_depth": [None, 3, 5, 10, 15, 20],
    "dt__min_samples_split": [2, 5, 10, 20],
    "dt__min_samples_leaf": [1, 2, 5, 10],
    "dt__max_features": [None, "sqrt", "log2"]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

predictions = grid.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

Best parameters:
{'dt__criterion': 'gini', 'dt__max_depth': 5, 'dt__max_features': None, 'dt__min_samples_leaf': 5, 'dt__min_samples_split': 2}
Best cross-validation accuracy:
0.7404024767801858
Test accuracy: 0.7478


**Model Engineering / AdaBoost**

In [8]:
pipe = Pipeline([("ada", AdaBoostClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42))])

params = {
    "ada__n_estimators": [25, 50, 100, 200, 500],
    "ada__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "ada__estimator__max_depth": [1, 2, 3],
    "ada__estimator__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

predictions = grid.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

Best parameters:
{'ada__estimator__class_weight': 'balanced', 'ada__estimator__max_depth': 3, 'ada__learning_rate': 0.05, 'ada__n_estimators': 500}
Best cross-validation accuracy:
0.7587203302373581
Test accuracy: 0.7583


**Model Engineering / XGBoost**

In [9]:
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')
bst.fit(X_train, y_train)
preds = bst.predict(X_test)


pipe = Pipeline([
    ("xgb", XGBClassifier(
        n_estimators=2,
        max_depth=2,
        learning_rate=1,
        objective="binary:logistic"
    ))
])

params = {
    "xgb__n_estimators": [100, 200, 300, 500],
    "xgb__learning_rate": [0.01, 0.05, 0.1, 0.3],
    "xgb__max_depth": [2, 3, 4, 6, 8],
    "xgb__subsample": [0.6, 0.8, 1.0]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

predictions = grid.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

Best parameters:
{'xgb__learning_rate': 0.01, 'xgb__max_depth': 4, 'xgb__n_estimators': 200, 'xgb__subsample': 0.8}
Best cross-validation accuracy:
0.7591847265221878
Test accuracy: 0.7579


**Model Engineering / RandomForest**

In [10]:
pipe = Pipeline([
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ))
])

params = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_leaf": [1, 3, 5, 10]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

predictions = grid.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

Best parameters:
{'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 10, 'rf__n_estimators': 200}
Best cross-validation accuracy:
0.7598555211558308
Test accuracy: 0.7585
